# Project Sanjay MK2 -- Day 3: Full Data Sweep & Retrain as police_full_v2

**Problem:** Day 2 model (`best_day2.pt`, YOLO11s, 100 epochs) scored weapon_person mAP50=0.019.
Root cause: all weapon_person training data was synthetic. Also: explosive_device has ZERO data,
fire/person data lacks aerial perspective.

**Fix:** Download real data from multiple free sources, remove synthetic weapon data,
merge, and retrain as `police_full_v2` using `best_day2.pt` weights as initialization.

| Class | New Source | Est. Images |
|-------|-----------|-------------|
| weapon_person (1) | OpenImages + YouTube-GDD + Kaggle | ~8,500+ |
| explosive_device (4) | Roboflow ZIPs (manual download) | ~1,000-3,000 |
| fire (3) | Kaggle fire/smoke aerial | ~2,000+ |
| person+vehicle (0,2) | HIT-UAV thermal aerial | ~2,898 |

| Item | Detail |
|------|--------|
| Checkpoint | `best_day2.pt` (YOLO11s, mAP50=0.593) |
| Training | 75 epochs, fresh optimizer (NOT resume) |
| Target | weapon_person mAP50 > 0.10, no regression on others |

## CELL 1 — Install + clone repo

In [ ]:
!pip install ultralytics kagglehub scipy pillow requests tqdm -q
!git clone https://github.com/YOUR_ORG/Sanjay_MK2 /content/Sanjay_MK2
%cd /content/Sanjay_MK2

# Patch YAML with absolute Colab path (Ultralytics resolves relative paths
# from its own datasets dir, not the project root)
import os
os.makedirs('config/training', exist_ok=True)
with open('config/training/visdrone_police.yaml', 'w') as f:
    f.write(
        "path: /content/Sanjay_MK2/data/visdrone_police\n"
        "train: images/train\n"
        "val: images/val\n"
        "test: images/test\n"
        "names:\n"
        "  0: person\n"
        "  1: weapon_person\n"
        "  2: vehicle\n"
        "  3: fire\n"
        "  4: explosive_device\n"
        "  5: crowd\n"
    )
print('YAML written with absolute path')

# Verify GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## CELL 2 — Mount Drive + copy checkpoint

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

# Copy best_day2.pt from Google Drive
# Adjust path if your Drive layout differs
SRC_CKPT = '/content/drive/MyDrive/SanjayMK2/runs/police_full_v1/weights/best.pt'
DST_CKPT = '/content/Sanjay_MK2/best_day2.pt'

if os.path.exists(SRC_CKPT):
    shutil.copy(SRC_CKPT, DST_CKPT)
    print(f'Checkpoint copied: {os.path.getsize(DST_CKPT) / 1e6:.1f} MB')
else:
    print(f'WARNING: {SRC_CKPT} not found!')
    print('Upload best_day2.pt manually or adjust the path above.')

## CELL 3 — VisDrone download + remap (skip if cached)

In [ ]:
from pathlib import Path

vd_train = Path('data/visdrone_police/images/train')
if vd_train.exists() and len(list(vd_train.glob('*'))) > 6000:
    print(f'VisDrone already set up ({len(list(vd_train.glob("*")))} train images). Skipping.')
else:
    !python scripts/train_yolo.py --setup-visdrone

## CELL 4 — D-Fire from Kaggle (skip if cached)

In [ ]:
from pathlib import Path

dfire_train = Path('data/supplementary/fire_dfire/images/train')
if dfire_train.exists() and len(list(dfire_train.glob('*'))) > 1000:
    print(f'D-Fire already set up ({len(list(dfire_train.glob("*")))} train images). Skipping.')
else:
    import kagglehub, shutil
    from pathlib import Path

    path = kagglehub.dataset_download('sayedgamal99/smoke-fire-detection-yolo')
    FIRE_SRC = Path(path) / 'data'
    FIRE_DST = Path('data/supplementary/fire_dfire')

    for ss, sd in [('train','train'), ('val','val'), ('test','val')]:
        sl, si = FIRE_SRC/ss/'labels', FIRE_SRC/ss/'images'
        dl, di = FIRE_DST/'labels'/sd, FIRE_DST/'images'/sd
        dl.mkdir(parents=True, exist_ok=True)
        di.mkdir(parents=True, exist_ok=True)
        if not sl.exists(): continue
        for lp in sorted(sl.glob('*.txt')):
            lines = []
            with open(lp) as f:
                for line in f:
                    p = line.strip().split()
                    if len(p) >= 5:
                        p[0] = '3'  # smoke + fire both -> class 3
                        lines.append(' '.join(p))
            with open(dl/lp.name, 'w') as f:
                f.write('\n'.join(lines) + '\n' if lines else '')
            stem = lp.stem
            for ext in ['.jpg','.jpeg','.png']:
                isrc = si/(stem+ext)
                if isrc.exists():
                    idst = di/(stem+ext)
                    if not idst.exists(): shutil.copy2(isrc, idst)
                    break
    print('D-Fire done — train:', len(list((FIRE_DST/'images'/'train').glob('*'))),
          'val:', len(list((FIRE_DST/'images'/'val').glob('*'))))

## CELL 5 — ShanghaiTech crowd from Kaggle (skip if cached)

In [ ]:
from pathlib import Path

crowd_train = Path('data/supplementary/crowd_shanghaitech/images/train')
if crowd_train.exists() and len(list(crowd_train.glob('*'))) > 100:
    print(f'ShanghaiTech already set up ({len(list(crowd_train.glob("*")))} train images). Skipping.')
else:
    import kagglehub, shutil, numpy as np
    from pathlib import Path
    from scipy.io import loadmat
    from PIL import Image

    path = kagglehub.dataset_download('tthien/shanghaitech')
    CS = Path(path) / 'ShanghaiTech' / 'part_A'
    CD = Path('data/supplementary/crowd_shanghaitech')

    for ss, sd in [('train_data','train'), ('test_data','val')]:
        ig = CS/ss/'images'; gt = CS/ss/'ground-truth'
        di = CD/'images'/sd; dl = CD/'labels'/sd
        di.mkdir(parents=True, exist_ok=True); dl.mkdir(parents=True, exist_ok=True)
        if not ig.exists(): continue
        c = 0
        for ip in sorted(ig.glob('*.jpg')):
            gp = gt/f'GT_{ip.stem}.mat'
            if not gp.exists(): continue
            try:
                mat = loadmat(str(gp))
                pts = mat['image_info'][0][0][0][0][0]
                if len(pts) < 5: continue
                im = Image.open(ip); w, h = im.size
                xn = max(0, pts[:,0].min()-20)/w; yn = max(0, pts[:,1].min()-20)/h
                xx = min(w, pts[:,0].max()+20)/w; yx = min(h, pts[:,1].max()+20)/h
                cx=(xn+xx)/2; cy=(yn+yx)/2; bw=xx-xn; bh=yx-yn
                shutil.copy2(ip, di/ip.name)
                with open(dl/f'{ip.stem}.txt','w') as f:
                    f.write(f'5 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n')
                c += 1
            except: continue
        print(f'ShanghaiTech {ss}->{sd}: {c} images')

## CELL 6 -- Download ALL automated supplementary data

Downloads from 5 free sources in one command:
1. **OpenImages v7 Handgun** (~561 images) -- weapon_person
2. **YouTube-GDD** (~5,000 images) -- weapon_person
3. **Kaggle gun+handgun** (~6,000 images) -- weapon_person
4. **HIT-UAV thermal** (~2,898 images) -- aerial person + vehicle (thermal IR, 60-130m altitude)
5. **Kaggle fire/smoke** (varies) -- aerial fire supplement

In [ ]:
# Download ALL automated supplementary sources in one command
# Covers: weapons (OpenImages + YouTube-GDD + Kaggle) + HIT-UAV thermal + aerial fire
!python scripts/prepare_supplementary_data.py --supplement-all --max-images 5000

# Show what we got
import os
from pathlib import Path
for d in sorted(Path('data/supplementary').iterdir()):
    if d.is_dir() and not d.name.startswith('_'):
        train = d / 'images' / 'train'
        n = len(list(train.glob('*'))) if train.exists() else 0
        print(f'  {d.name}: {n} train images')

In [ ]:
# SKIP THIS CELL if you didn't download Roboflow ZIPs.
# Uncomment and adjust paths for each ZIP you downloaded.

from pathlib import Path

# Example: Import TrashIED (class 4 = explosive_device)
# !python scripts/prepare_supplementary_data.py \
#     --import-roboflow-zip /content/TrashIED.zip \
#     --import-class 4 --import-name explosive_trashied

# Example: Import Grenade/Landmine (class 4 = explosive_device)
# !python scripts/prepare_supplementary_data.py \
#     --import-roboflow-zip /content/Grenade.zip \
#     --import-class 4 --import-name explosive_grenade

# Example: Import Abandoned Bags from drone perspective (class 4 = explosive_device)
# !python scripts/prepare_supplementary_data.py \
#     --import-roboflow-zip /content/AbandonedBags.zip \
#     --import-class 4 --import-name explosive_abandoned_bags

# Check what explosive data we have
explosive_dirs = [d for d in Path('data/supplementary').iterdir()
                  if d.is_dir() and 'explosive' in d.name]
if explosive_dirs:
    for d in explosive_dirs:
        n = len(list((d/'images'/'train').glob('*'))) if (d/'images'/'train').exists() else 0
        print(f'  {d.name}: {n} train images')
else:
    print('  No explosive_device data yet (optional -- skip for Day 3 if needed)')

## CELL 6b -- (OPTIONAL) Import explosive_device from Roboflow

The explosive_device class has **ZERO** training data. To fix this, manually download
YOLO-format ZIPs from Roboflow Universe (free account, no API key):

1. Go to [TrashIED](https://universe.roboflow.com/mda522/trashied) (~2,066 images, IED+Trash)
2. Click "Download Dataset" -> select "YOLOv8" format -> download ZIP
3. Upload the ZIP to Colab (or Google Drive)
4. Run the cell below with the correct path

Other good sources for explosive_device:
- [Grenade/Landmine/Explosive](https://universe.roboflow.com/incheonairport/grenade-umqpw) (~933 images)
- [Explosive](https://universe.roboflow.com/volodymyr-uretn/explosive-5enu5) (~440 images, Bomb+Mines)
- [Abandoned Bags - Drone](https://universe.roboflow.com/drone-analysis/abandoned-bags-2afuo) (~840 images, drone perspective!)

## CELL 7 -- Remove synthetic weapons + Merge all + Audit

1. Remove old synthetic weapon files from visdrone_police/
2. Clear stale merged data
3. Merge all supplementary (weapon_openimages + weapon_youtube_gdd + weapon_kaggle + fire_dfire + crowd_shanghaitech)
4. Merge into training set
5. Audit final dataset

In [ ]:
import shutil
from pathlib import Path

# Step 1: Remove old synthetic weapon data from visdrone_police
!python scripts/prepare_supplementary_data.py --remove-synthetic-weapons

# Step 2: Clear stale supplementary_merged to rebuild from scratch
merged = Path('data/supplementary_merged')
if merged.exists():
    shutil.rmtree(merged)
    print(f'Cleared stale {merged}')

# Step 3: Merge all supplementary sources (weapon_openimages + fire_dfire + crowd_shanghaitech)
!python scripts/prepare_supplementary_data.py --merge-all

# Step 4: Remove old supplementary_merged files from visdrone_police to avoid duplicates
vd = Path('data/visdrone_police')
removed = 0
for subdir in ['images', 'labels']:
    for split in ['train', 'val']:
        for f in (vd/subdir/split).glob('supplementarymerged_*'):
            f.unlink()
            removed += 1
print(f'Removed {removed} old merged files from visdrone_police')

# Step 5: Merge new supplementary data into training set
!python scripts/train_yolo.py --merge data/supplementary_merged --auto-prefix

# Step 6: Audit
!python scripts/audit_dataset.py data/visdrone_police

## CELL 8 -- Train police_full_v2 (75 epochs, auto-resumable)

Results save to **Google Drive** -- safe against Colab disconnects.

**If training is interrupted:** Just re-run this cell. It auto-detects `last.pt` on Drive
and resumes from where it left off (optimizer state, epoch counter, LR schedule all preserved).

**First run:** Uses `best_day2.pt` weights as initialization with a fresh optimizer.
**Subsequent runs:** Resumes from the last saved checkpoint.

In [ ]:
import os
from ultralytics import YOLO

DRIVE_PROJECT = '/content/drive/MyDrive/SanjayMK2/runs'
RUN_NAME = 'police_full_v2'
LAST_PT = f'{DRIVE_PROJECT}/{RUN_NAME}/weights/last.pt'

# Auto-detect: resume from last.pt if a previous run exists on Drive
if os.path.exists(LAST_PT):
    print(f'Resuming from: {LAST_PT}')
    print(f'  Checkpoint size: {os.path.getsize(LAST_PT) / 1e6:.1f} MB')
    model = YOLO(LAST_PT)
    results = model.train(
        resume=True,  # continues from saved epoch, optimizer state, LR schedule
    )
else:
    print('Starting fresh training from best_day2.pt')
    model = YOLO('best_day2.pt')
    results = model.train(
        data='config/training/visdrone_police.yaml',
        epochs=75,
        imgsz=640,
        device=0,
        batch=-1,           # auto batch
        patience=20,
        name=RUN_NAME,
        project=DRIVE_PROJECT,
        exist_ok=True,
        # Aerial augmentations (same as Day 2)
        degrees=15.0,
        flipud=0.5,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.1,
        scale=0.5,
        translate=0.1,
        hsv_h=0.015,
        hsv_s=0.4,
        hsv_v=0.4,
    )

print('Best mAP50:', results.results_dict.get('metrics/mAP50(B)'))
print(f'Weights on Drive: {DRIVE_PROJECT}/{RUN_NAME}/weights/')

## Day 3 Success Bar

| Metric | Day 2 (baseline) | Day 3 Target |
|--------|------------------|--------------|
| mAP50 (all) | 0.593 | > 0.55 (no regression) |
| person mAP50 | 0.440 | > 0.35 |
| vehicle mAP50 | 0.731 | > 0.65 |
| fire mAP50 | 0.780 | > 0.40 |
| crowd mAP50 | 0.995 | > 0.15 |
| **weapon_person mAP50** | **0.019** | **> 0.10** |
| explosive_device mAP50 | -- | > 0.00 if data added |

## What changed from Day 2

- **Removed:** 800+100 synthetic weapon images (VisDrone person relabels)
- **Added (weapons):** ~8,500+ real weapon images from 3 free sources:
  - OpenImages v7 Handgun (~561 images)
  - YouTube-GDD (~5,000 images, gun+person from YouTube videos)
  - Kaggle gun+handgun (~6,000 images, CC0 license)
- **Added (thermal aerial):** ~2,898 HIT-UAV thermal IR images (person+vehicle, 60-130m drone)
- **Added (aerial fire):** Kaggle fire/smoke detection dataset
- **Added (explosives, optional):** Roboflow IED/grenade/bomb ZIPs
- **Training:** Fresh optimizer from Day 2 weights (not resume), 75 epochs

## Data sources reference

| Source | Class | Images | License |
|--------|-------|--------|---------|
| OpenImages v7 | weapon_person | ~561 | CC BY |
| YouTube-GDD | weapon_person | ~5,000 | Academic |
| Kaggle gundetection | weapon_person | ~3,000 | CC0 |
| Kaggle handgun-detection | weapon_person | ~2,900 | CC0 |
| HIT-UAV | person, vehicle | ~2,898 | CC BY 4.0 |
| Kaggle fire/smoke | fire | varies | check |
| Roboflow TrashIED | explosive_device | ~2,066 | CC BY 4.0 |
| Roboflow Grenade | explosive_device | ~933 | CC BY 4.0 |

In [ ]:
from google.colab import files
import os

best_path = '/content/drive/MyDrive/SanjayMK2/runs/police_full_v2/weights/best.pt'
if os.path.exists(best_path):
    print(f'Best weights: {os.path.getsize(best_path) / 1e6:.1f} MB')
    files.download(best_path)
    print('Downloaded best.pt')
else:
    print(f'WARNING: {best_path} not found. Check training output above.')

print()
print('Next steps:')
print('  1. Copy best.pt to: runs/detect/runs/detect/police_full_v2/weights/best.pt')
print('  2. Validate: python scripts/validate_model.py --yolo best.pt --all --compare')
print('  3. Check weapon_person mAP50 > 0.10 and no regression on other classes')

## Day 3 Success Bar

| Metric | Day 2 (baseline) | Day 3 Target |
|--------|------------------|--------------|
| mAP50 (all) | 0.593 | > 0.55 (no regression) |
| person mAP50 | 0.440 | > 0.35 |
| vehicle mAP50 | 0.731 | > 0.65 |
| fire mAP50 | 0.780 | > 0.40 |
| crowd mAP50 | 0.995 | > 0.15 |
| **weapon_person mAP50** | **0.019** | **> 0.10** |
| explosive_device mAP50 | -- | -- (deferred) |

## What changed from Day 2

- **Removed:** 800+100 synthetic weapon images (VisDrone person relabels)
- **Added:** ~8,500+ real weapon images from 3 free sources:
  - OpenImages v7 Handgun (~561 images, ground-truth bboxes)
  - YouTube-GDD (~5,000 images, gun+person from YouTube videos)
  - Kaggle gun+handgun (~6,000 images, CC0 license)
- **Training:** Fresh optimizer from Day 2 weights (not resume), 75 epochs
- **Expected impact:** weapon_person should improve dramatically; other classes unchanged

## Data sources reference

| Source | Images | Format | License |
|--------|--------|--------|---------|
| OpenImages v7 | ~561 | Direct CSV+S3 download | CC BY |
| YouTube-GDD | ~5,000 | YOLO (GitHub) | Academic |
| Kaggle gundetection | ~3,000 | YOLO (kagglehub) | CC0 |
| Kaggle handgun-detection | ~2,900 | VOC XML (kagglehub) | CC0 |